In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install timm scikit-learn seaborn

In [ ]:
import os
import torch
import timm
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import *
from torch.amp import autocast
from torch.cuda.amp import GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

Using device: cuda


In [ ]:
data_dir = "/content/drive/MyDrive/AMD_combined"
results_path = "/content/drive/MyDrive/amd_results"
os.makedirs(results_path, exist_ok=True)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(os.path.join(data_dir,"train"), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir,"val"), transform=val_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir,"test"), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
train_labels = [label for _, label in train_dataset.samples]
num_neg = train_labels.count(0)
num_pos = train_labels.count(1)

pos_weight = torch.tensor([num_neg / num_pos]).to(device)
print("Class weight (pos_weight):", pos_weight.item())

Class weight (pos_weight): 0.682539701461792


In [ ]:
model = timm.create_model("densenet121", pretrained=True)
in_features = model.classifier.in_features
model.classifier = torch.nn.Linear(in_features, 1)
model = model.to(device)

# Freeze first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze classifier initially
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

scaler = GradScaler()

/tmp/ipython-input-2404624162.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
best_auc = 0
patience = 5
counter = 0
num_epochs = 20

for epoch in range(num_epochs):

    # Unfreeze full model after 3 epochs
    if epoch == 3:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        with autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)

            preds.extend(probs.cpu().numpy())
            targets.extend(labels.numpy())

    val_auc = roc_auc_score(targets, preds)

    print(f"Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f}, Val AUC={val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f"{results_path}/best_model_combined.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

Epoch 1: Loss=0.3672, Val AUC=0.9999
Epoch 2: Loss=0.2628, Val AUC=1.0000
Epoch 3: Loss=0.2355, Val AUC=0.9999
Epoch 4: Loss=0.0893, Val AUC=1.0000
Epoch 5: Loss=0.0514, Val AUC=1.0000
Epoch 6: Loss=0.0452, Val AUC=1.0000
Epoch 7: Loss=0.0387, Val AUC=1.0000
Epoch 8: Loss=0.0310, Val AUC=1.0000
Epoch 9: Loss=0.0294, Val AUC=1.0000
Early stopping triggered


In [ ]:
model.load_state_dict(torch.load(f"{results_path}/best_model_combined.pth"))
model.eval()

preds, targets = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)

        preds.extend(probs.cpu().numpy())
        targets.extend(labels.numpy())

binary_preds = [1 if p>0.5 else 0 for p in preds]

accuracy = accuracy_score(targets, binary_preds)
precision = precision_score(targets, binary_preds)
recall = recall_score(targets, binary_preds)
f1 = f1_score(targets, binary_preds)
auc = roc_auc_score(targets, preds)

cm = confusion_matrix(targets, binary_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn/(tn+fp)

print("Final Test Results")
print("Accuracy:", accuracy)
print("Sensitivity:", recall)
print("Specificity:", specificity)
print("F1:", f1)
print("AUC:", auc)
print("Confusion Matrix:\n", cm)

# Save metrics
pd.DataFrame({
    "Accuracy":[accuracy],
    "Sensitivity":[recall],
    "Specificity":[specificity],
    "F1":[f1],
    "AUC":[auc]
}).to_csv(f"{results_path}/combined_metrics.csv", index=False)

Final Test Results
Accuracy: 1.0
Sensitivity: 1.0
Specificity: 1.0
F1: 1.0
AUC: 1.0
Confusion Matrix:
 [[350   0]
 [  0 350]]


In [ ]:
###The above code is not evaluated in OCT2017/test, it is evaluated in AMD_combined/test, which is not necessay as it gives 100% accuracy either way

NameError: name 'OCTBinaryDataset' is not defined

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
from torchvision import transforms
import torch
from sklearn.metrics import *

class OCTBinaryDataset(Dataset):
    def __init__(self, root, transform=None):
        self.dataset = ImageFolder(root)
        self.transform = transform

        self.valid_classes = ["NORMAL", "DRUSEN", "CNV"]

        self.samples = [
            (path, self.map_label(label))
            for path, label in self.dataset.samples
            if self.dataset.classes[label] in self.valid_classes
        ]

    def map_label(self, label):
        class_name = self.dataset.classes[label]
        if class_name == "NORMAL":
            return 0
        else:
            return 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = self.dataset.loader(path)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
import timm

model = timm.create_model("densenet121", pretrained=False)
in_features = model.classifier.in_features
model.classifier = torch.nn.Linear(in_features, 1)

model.load_state_dict(torch.load("/content/drive/MyDrive/amd_results/best_model_combined.pth", map_location=device))
model = model.to(device)
model.eval()

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNormAct2d(
      64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): DenseBlock(
      (denselayer1): DenseLayer(
        (norm1): BatchNormAct2d(
          64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNormAct2d(
          128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
  

In [ ]:
from torch.utils.data import DataLoader

oct2017_path = "/content/drive/MyDrive/OCT2017/test"

oct_test_dataset = OCTBinaryDataset(
    oct2017_path,
    transform=val_transforms
)

oct_test_loader = DataLoader(
    oct_test_dataset,
    batch_size=32,
    shuffle=False
)

print("External test size:", len(oct_test_dataset))

preds = []
targets = []

with torch.no_grad():
    for images, labels in oct_test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)

        preds.extend(probs.cpu().numpy())
        targets.extend(labels.numpy())

binary_preds = [1 if p>0.5 else 0 for p in preds]

accuracy = accuracy_score(targets, binary_preds)
recall = recall_score(targets, binary_preds)
auc = roc_auc_score(targets, preds)

cm = confusion_matrix(targets, binary_preds)
tn, fp, fn, tp = cm.ravel()
specificity = tn/(tn+fp)

print("\nOCT2017 External Results")
print("Accuracy:", accuracy)
print("Sensitivity:", recall)
print("Specificity:", specificity)
print("AUC:", auc)
print("Confusion Matrix:\n", cm)

External test size: 750

OCT2017 External Results
Accuracy: 0.9986666666666667
Sensitivity: 0.998
Specificity: 1.0
AUC: 0.999984
Confusion Matrix:
 [[250   0]
 [  1 499]]
